# Análisis Exploratorio de Datos (EDA) — GP de Abu Dhabi 2021

Este notebook analiza la carrera del Gran Premio de Abu Dhabi 2021 (la famosa definición del título entre **Max Verstappen** y **Lewis Hamilton**) utilizando datos oficiales de telemetría mediante la librería **FastF1**.

El objetivo principal es visualizar la **diferencia de tiempo por vuelta** entre ambos pilotos y resaltar el período de **Safety Car** que cambió el desenlace del campeonato.

## 1. Importación de Librerías y Configuración

In [ ]:
# ---------- Librerías ----------
import fastf1                       # Telemetría y datos oficiales de F1
import pandas as pd                 # Manipulación de datos
import numpy as np                  # Operaciones numéricas
import matplotlib.pyplot as plt     # Visualización
import matplotlib.ticker as ticker  # Formato de ejes

# Activar caché para no re-descargar datos en cada ejecución
fastf1.Cache.enable_cache('cache')

print('Librerías cargadas correctamente ✓')

## 2. Carga de Datos de la Sesión

Cargamos la **carrera** (`'R'`) del GP de Abu Dhabi 2021 y filtramos las vueltas de Verstappen (`VER`) y Hamilton (`HAM`).

In [ ]:
# ---------- Parámetros de la sesión ----------
YEAR        = 2021
GRAND_PRIX  = 'Abu Dhabi'
SESSION     = 'R'            # 'R' = Race, 'Q' = Qualifying, 'FP1/FP2/FP3'

# Cargar sesión (la primera vez descarga ~30 MB, luego usa caché)
session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION)
session.load()

print(f'Sesión cargada: {session.event["EventName"]} {YEAR} — {session.name}')
print(f'Total de vueltas de carrera: {session.total_laps}')

## 3. Preparación de Datos

In [ ]:
# ---------- Filtrar vueltas de VER y HAM ----------
# pick_driver() devuelve un DataFrame con los laps del piloto indicado
# pick_quicklaps() filtra vueltas anómalas (pit-in/out, vuelta 1, etc.)
ver = session.laps.pick_driver('VER').pick_quicklaps().reset_index()
ham = session.laps.pick_driver('HAM').pick_quicklaps().reset_index()

# Unir ambos DataFrames por número de vuelta para poder compararlos
merged = ver.merge(ham, on='LapNumber', suffixes=('_VER', '_HAM'))

# Calcular la diferencia de tiempo por vuelta (en segundos):
#   > 0  →  Hamilton fue más lento en esa vuelta
#   < 0  →  Verstappen fue más lento en esa vuelta
merged['gap_s'] = (
    merged['LapTime_HAM'].dt.total_seconds()
    - merged['LapTime_VER'].dt.total_seconds()
)

print(f'Vueltas comparables (después de filtrar): {len(merged)}')
merged[['LapNumber', 'gap_s']].head(10)

## 4. Identificación del período de Safety Car

En Abu Dhabi 2021 hubo un Safety Car tardío provocado por el accidente de **Nicholas Latifi** en la vuelta 53. El SC terminó en la vuelta 57 (última vuelta de carrera con reinicio).

In [ ]:
# ---------- Detectar vueltas bajo Safety Car ----------
# La columna 'TrackStatus' contiene flags (ej. '4' = SC, '6'/'7' = VSC)
# Intentamos detectarlo automáticamente; si no, usamos los valores conocidos.

all_laps = session.laps

# Buscar vueltas donde el TrackStatus indique Safety Car
sc_laps = all_laps[all_laps['TrackStatus'].str.contains('4|5|6|7', na=False)]

if not sc_laps.empty:
    SC_START = int(sc_laps['LapNumber'].min())
    SC_END   = int(sc_laps['LapNumber'].max())
    print(f'Safety Car detectado automáticamente: vueltas {SC_START} a {SC_END}')
else:
    # Valores conocidos de Abu Dhabi 2021
    SC_START = 53
    SC_END   = 57
    print(f'Safety Car (valores históricos): vueltas {SC_START} a {SC_END}')

## 5. Gráfica — Diferencia de Tiempo por Vuelta VER vs HAM

Visualizamos la diferencia de ritmo vuelta a vuelta entre Verstappen y Hamilton. La **banda sombreada en amarillo** indica el período bajo Safety Car.

In [ ]:
# =====================================================================
# Paleta de colores inspirada en F1
# =====================================================================
F1_RED      = '#E10600'   # Rojo F1 / Red Bull
MERC_TEAL   = '#00D2BE'   # Verde azulado Mercedes
SC_YELLOW   = '#FFD700'   # Amarillo Safety Car
BG_DARK     = '#1a1a2e'   # Fondo oscuro
GRID_COLOR  = '#333355'   # Líneas de cuadrícula
TEXT_COLOR  = '#e0e0e0'   # Texto claro

# =====================================================================
# Crear la figura
# =====================================================================
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_DARK)

# ---------- Banda sombreada del Safety Car ----------
ax.axvspan(
    SC_START - 0.5, SC_END + 0.5,
    color=SC_YELLOW, alpha=0.15,
    label='Safety Car'
)
# Línea vertical al inicio y fin del SC para mayor claridad
ax.axvline(SC_START, color=SC_YELLOW, linewidth=0.8, linestyle='--', alpha=0.6)
ax.axvline(SC_END,   color=SC_YELLOW, linewidth=0.8, linestyle='--', alpha=0.6)

# ---------- Línea base en 0 (ritmo idéntico) ----------
ax.axhline(0, color=GRID_COLOR, linewidth=1, linestyle='-')

# ---------- Curva de diferencia de tiempo ----------
ax.plot(
    merged['LapNumber'], merged['gap_s'],
    color=F1_RED, linewidth=2.2, zorder=3
)

# ---------- Relleno: zona donde HAM es más lento (gap > 0) ----------
ax.fill_between(
    merged['LapNumber'], merged['gap_s'], 0,
    where=(merged['gap_s'] > 0),
    color=F1_RED, alpha=0.20, interpolate=True,
    label='HAM más lento'
)

# ---------- Relleno: zona donde VER es más lento (gap < 0) ----------
ax.fill_between(
    merged['LapNumber'], merged['gap_s'], 0,
    where=(merged['gap_s'] <= 0),
    color=MERC_TEAL, alpha=0.20, interpolate=True,
    label='VER más lento'
)

# =====================================================================
# Estilo de ejes y etiquetas
# =====================================================================
ax.set_xlabel('Vuelta', fontsize=12, color=TEXT_COLOR, labelpad=10)
ax.set_ylabel(
    'Δ Tiempo por vuelta (s)\nHAM más lento ↑  ·  VER más lento ↓',
    fontsize=11, color=TEXT_COLOR, labelpad=10
)
ax.set_title(
    'Abu Dhabi 2021 — Ritmo por vuelta: Verstappen vs. Hamilton',
    fontsize=15, fontweight='bold', color='white', pad=15
)

# Cuadrícula sutil
ax.grid(color=GRID_COLOR, linewidth=0.4, alpha=0.5)

# Color de ticks y bordes
ax.tick_params(colors=TEXT_COLOR, labelsize=10)
for spine in ax.spines.values():
    spine.set_color(GRID_COLOR)

# Leyenda
legend = ax.legend(
    loc='upper left', frameon=True, framealpha=0.3,
    facecolor=BG_DARK, edgecolor=GRID_COLOR,
    fontsize=10, labelcolor=TEXT_COLOR
)

# Anotación del incidente de Latifi
ax.annotate(
    'Accidente de\nLatifi → SC',
    xy=(SC_START, 0), xytext=(SC_START - 8, ax.get_ylim()[1] * 0.7),
    fontsize=9, color=SC_YELLOW, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=SC_YELLOW, lw=1.5),
    ha='center'
)

fig.tight_layout()
plt.show()

# Guardar en la carpeta de visualisaciones
fig.savefig(
    '../visualisations/01_gap_ver_vs_ham_abu_dhabi_2021.png',
    dpi=200, bbox_inches='tight', facecolor=BG_DARK
)
print('Gráfica guardada en visualisations/01_gap_ver_vs_ham_abu_dhabi_2021.png ✓')

## 6. Análisis de las últimas 10 vueltas (Tiempos por vuelta)

Aislamos las últimas 10 vueltas de la carrera para ver en detalle cómo impactó el Safety Car a los tiempos de vuelta de ambos contendientes.

In [ ]:
# ---------- Aislar las últimas 10 vueltas ----------
last_10_ver = ver.tail(10).copy()
last_10_ham = ham.tail(10).copy()

# Convertir el tiempo de vuelta a segundos para graficar fácilmente
last_10_ver['LapTime_s'] = last_10_ver['LapTime'].dt.total_seconds()
last_10_ham['LapTime_s'] = last_10_ham['LapTime'].dt.total_seconds()

# =====================================================================
# Gráfica de Líneas: Tiempos de Vuelta en los últimos compases
# =====================================================================
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_DARK)

# Banda del Safety Car
ax.axvspan(
    SC_START - 0.5, SC_END + 0.5,
    color=SC_YELLOW, alpha=0.15,
    label='Período de Safety Car'
)

# Graficar los tiempos de vuelta
ax.plot(
    last_10_ver['LapNumber'], last_10_ver['LapTime_s'],
    color=F1_RED, marker='o', linewidth=2.5, markersize=6, label='VER (Red Bull)', zorder=4
)
ax.plot(
    last_10_ham['LapNumber'], last_10_ham['LapTime_s'],
    color=MERC_TEAL, marker='s', linewidth=2.5, markersize=6, label='HAM (Mercedes)', zorder=4
)

# Estilos
ax.set_xlabel('Vuelta', fontsize=12, color=TEXT_COLOR, labelpad=10)
ax.set_ylabel('Tiempo de Vuelta (segundos)', fontsize=12, color=TEXT_COLOR, labelpad=10)
ax.set_title(
    'Evolución del Tiempo de Vuelta - Últimas 10 Vueltas (Abu Dhabi 2021)',
    fontsize=14, fontweight='bold', color='white', pad=15
)

ax.grid(color=GRID_COLOR, linewidth=0.5, alpha=0.6)
ax.tick_params(colors=TEXT_COLOR, labelsize=10)
for spine in ax.spines.values():
    spine.set_color(GRID_COLOR)

ax.legend(loc='upper right', facecolor=BG_DARK, edgecolor=GRID_COLOR, labelcolor=TEXT_COLOR)

fig.tight_layout()
plt.show()

# Guardar imagen
fig.savefig(
    '../visualisations/02_last10laps_ver_vs_ham_abu_dhabi_2021.png',
    dpi=200, bbox_inches='tight', facecolor=BG_DARK
)
print('Gráfica guardada en visualisations/02_last10laps_ver_vs_ham_abu_dhabi_2021.png ✓')